# Task 4 — Healthcare Coverage & Financing Data Preparation
المخرجات: `healthcare_coverage_clean.csv` و `health_financing_clean.csv` و `cleaning_log.txt` (+ `coverage_summary.csv`).

قواعد ثابتة في النوتبوك: لا `fillna(0)` ولا Interpolation ولا Fill، ولا تعديل على الـRaw Files، ولا Mapping يدوي للدول، والربط دائمًا على `ISO3 + Year`، وأي ISO3 مش موجود في الـDictionary بيتسجل وبيتبعت للـLead.

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

RAW_DIR = Path(".")   # فولدر الملفات الأصلية
OUT_DIR = Path(".")   # فولدر المخرجات

UHC_FILE = RAW_DIR / "10_مؤشر_تغطية_الخدمات_الصحية_الشاملة.csv"
GHED_FILE = RAW_DIR / "09_مؤشرات_الإنفاق_والتمويل_الصحي.XLSX"
DICT_FILE = RAW_DIR / "country_dictionary_v1.csv"

# لو في قيم خارج النطاق: "stop" يوقف التشغيل عشان تتراجع الأول، "null" يحولها NaN ويسجلها
ON_OUT_OF_RANGE = "stop"

LOG = []
def log(msg=""):
    print(msg)
    LOG.append(str(msg))

def read_csv_safe(path):
    for enc in ("utf-8-sig", "utf-8", "cp1256", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue

def pick_col(df, candidates, label):
    lowered = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lowered:
            return lowered[cand.lower()]
    raise KeyError(f"Could not find the {label} column. Columns found: {list(df.columns)}")

def to_numeric_logged(series, name, label):
    num = pd.to_numeric(series, errors="coerce")
    failed = series.notna() & num.isna()
    if failed.any():
        log(f"[{label}] {name}: {int(failed.sum())} values could not be parsed as numbers (kept as NaN). Examples: {series[failed].head(5).tolist()}")
    return num

log("Files used:")
log(f"  - {UHC_FILE.name}")
log(f"  - {GHED_FILE.name} (sheet: Data)")
log(f"  - {DICT_FILE.name}")

Files used:
  - 10_مؤشر_تغطية_الخدمات_الصحية_الشاملة.csv
  - 09_مؤشرات_الإنفاق_والتمويل_الصحي.XLSX (sheet: Data)
  - country_dictionary_v1.csv


## 1) Country dictionary

In [2]:
cd_raw = read_csv_safe(DICT_FILE)
log("== country_dictionary_v1.csv ==")
log(f"Rows: {len(cd_raw)} | Columns: {list(cd_raw.columns)}")

d_iso = pick_col(cd_raw, ["ISO3", "iso_3", "iso3_code", "country_code", "code"], "ISO3 (dictionary)")
d_name = pick_col(cd_raw, ["Country", "country_name", "country_name_en", "name"], "Country name (dictionary)")
log(f"Using ISO3 column = '{d_iso}', Country column = '{d_name}'")

country_dict = cd_raw[[d_iso, d_name]].rename(columns={d_iso: "ISO3", d_name: "Country"})
country_dict["ISO3"] = country_dict["ISO3"].astype("string").str.strip().str.upper()
country_dict = country_dict.dropna(subset=["ISO3"]).drop_duplicates()

dup_dict = country_dict[country_dict["ISO3"].duplicated(keep=False)]
if len(dup_dict):
    log(dup_dict.to_string())
    raise ValueError("country_dictionary has the same ISO3 with different names. Send to the lead, do not fix manually.")
log(f"Dictionary ISO3 codes: {country_dict['ISO3'].nunique()}")

== country_dictionary_v1.csv ==
Rows: 196 | Columns: ['ISO3', 'M49', 'Country', 'WB_Name', 'WHO_Name', 'GHED_Name', 'Nurses_Source_Name', 'Region', 'Income_Group', 'In_GHED', 'In_WHO_Doctors', 'In_WorldBank', 'In_Nurses_Source']
Using ISO3 column = 'ISO3', Country column = 'Country'
Dictionary ISO3 codes: 196


## 2) Shared helpers (mapping + duplicates + summaries)

In [3]:
def attach_country(df, label, name_col=None):
    merged = df.merge(country_dict, on="ISO3", how="left", validate="m:1")
    bad = merged[merged["Country"].isna()]
    codes = bad["ISO3"].value_counts()
    n_codes = df["ISO3"].nunique()
    log(f"[{label}] ISO3 codes in data: {n_codes} | mapped to dictionary: {n_codes - len(codes)} | NOT in dictionary: {len(codes)}")
    if len(codes):
        log(f"[{label}] UNMAPPED ISO3 codes (rows each) -> send to the lead, do NOT map manually: {codes.to_dict()}")
        if name_col is not None:
            names = bad.drop_duplicates("ISO3").set_index("ISO3")[name_col].to_dict()
            log(f"[{label}] names of the unmapped codes in the raw file (reference only): {names}")
    return merged[merged["Country"].notna()].copy(), list(codes.index)

def resolve_duplicates(df, value_cols, label):
    key = ["ISO3", "Year"]
    n_dup = int(df.duplicated(key, keep=False).sum())
    if n_dup == 0:
        log(f"[{label}] Duplicate ISO3+Year rows: 0")
        return df
    log(f"[{label}] Duplicate ISO3+Year rows found: {n_dup}")
    deduped = df.drop_duplicates(subset=key + value_cols)
    log(f"[{label}] Fully identical duplicate rows removed: {len(df) - len(deduped)}")
    conflict = deduped[deduped.duplicated(key, keep=False)].sort_values(key)
    if len(conflict):
        log(conflict.head(30).to_string())
        raise ValueError(f"[{label}] Same ISO3+Year with DIFFERENT values. Review the cause before deleting anything.")
    return deduped

def summarize(df, cols, label):
    log(f"--- Validation summary: {label} ---")
    log(f"Rows: {len(df)} | Countries: {df['ISO3'].nunique()} | Duplicates on ISO3+Year: {int(df.duplicated(['ISO3', 'Year']).sum())}")
    for c in cols:
        yrs = df.loc[df[c].notna(), "Year"]
        log(f"{c}: non-null={int(df[c].notna().sum())} | missing={int(df[c].isna().sum())} | min year={yrs.min() if len(yrs) else None} | max year={yrs.max() if len(yrs) else None}")

def coverage_long(df, cols):
    parts = []
    for c in cols:
        s = (df.dropna(subset=[c]).groupby("Year")["ISO3"].nunique()
               .rename("Number_of_Countries_With_Data").reset_index())
        s.insert(0, "Indicator", c)
        parts.append(s)
    return pd.concat(parts, ignore_index=True)

## 3) Part 1 — UHC Service Coverage Index

In [4]:
log("")
log("== PART 1: UHC (file 10) ==")
uhc_raw = read_csv_safe(UHC_FILE)
log(f"Raw rows: {len(uhc_raw)} | Raw columns: {list(uhc_raw.columns)}")

c_code = pick_col(uhc_raw, ["SpatialDimValueCode", "ISO3", "COUNTRY_CODE", "Country Code", "CountryCode", "code"], "country code (UHC)")
c_year = pick_col(uhc_raw, ["Period", "Year", "TimeDim", "time"], "year (UHC)")
c_val = pick_col(uhc_raw, ["FactValueNumeric", "Value", "value", "UHC_Service_Coverage_Index"], "value (UHC)")
log(f"Columns used: code='{c_code}' -> ISO3 | year='{c_year}' -> Year | value='{c_val}' -> UHC_Service_Coverage_Index")

for ind_col in ("Indicator", "IndicatorCode"):
    if ind_col in uhc_raw.columns:
        log(f"Distinct values in '{ind_col}': {uhc_raw[ind_col].dropna().unique().tolist()[:10]}")

log(f"Raw distinct country codes: {uhc_raw[c_code].nunique()}")

uhc = pd.DataFrame({
    "ISO3": uhc_raw[c_code].astype("string").str.strip().str.upper(),
    "Year": to_numeric_logged(uhc_raw[c_year], "Year", "UHC"),
})

val_raw = uhc_raw[c_val]
num = pd.to_numeric(val_raw, errors="coerce")
if num.isna().sum() > val_raw.isna().sum():
    extracted = val_raw.astype("string").str.extract(r"^\s*(-?\d+(?:\.\d+)?)")[0]
    num = pd.to_numeric(extracted, errors="coerce")
    log("Value column contained text; leading number extracted.")
failed = val_raw.notna() & num.isna()
if failed.any():
    log(f"[UHC] {int(failed.sum())} values could not be parsed (kept NaN). Examples: {val_raw[failed].head(5).tolist()}")
uhc["UHC_Service_Coverage_Index"] = num

bad_key = uhc["ISO3"].isna() | uhc["Year"].isna()
log(f"[UHC] rows with missing ISO3 or Year (excluded): {int(bad_key.sum())}")
uhc = uhc[~bad_key].copy()
if ((uhc["Year"] % 1) != 0).any():
    raise ValueError("Non-integer Year values found in UHC; review.")
uhc["Year"] = uhc["Year"].astype("int64")

oor = uhc[(uhc["UHC_Service_Coverage_Index"] < 0) | (uhc["UHC_Service_Coverage_Index"] > 100)]
log(f"[UHC] values outside 0-100: {len(oor)}")
if len(oor):
    log(oor.head(30).to_string())
    if ON_OUT_OF_RANGE == "stop":
        raise ValueError("UHC values outside 0-100. Check they are not a parsing issue, then decide (see ON_OUT_OF_RANGE).")
    uhc.loc[oor.index, "UHC_Service_Coverage_Index"] = np.nan
    log("[UHC] out-of-range values set to NaN after review.")

uhc, uhc_unmapped = attach_country(uhc, "UHC")
uhc = resolve_duplicates(uhc, ["UHC_Service_Coverage_Index"], "UHC")

healthcare_coverage = (uhc[["ISO3", "Country", "Year", "UHC_Service_Coverage_Index"]]
                       .sort_values(["ISO3", "Year"]).reset_index(drop=True))
log(f"[UHC] rows before: {len(uhc_raw)} | after: {len(healthcare_coverage)}")
healthcare_coverage.head()


== PART 1: UHC (file 10) ==
Raw rows: 4680 | Raw columns: ['IndicatorCode', 'Indicator', 'ValueType', 'ParentLocationCode', 'ParentLocation', 'Location type', 'SpatialDimValueCode', 'Location', 'Period type', 'Period', 'IsLatestYear', 'Dim1 type', 'Dim1', 'Dim1ValueCode', 'Dim2 type', 'Dim2', 'Dim2ValueCode', 'Dim3 type', 'Dim3', 'Dim3ValueCode', 'DataSourceDimValueCode', 'DataSource', 'FactValueNumericPrefix', 'FactValueNumeric', 'FactValueUoM', 'FactValueNumericLowPrefix', 'FactValueNumericLow', 'FactValueNumericHighPrefix', 'FactValueNumericHigh', 'Value', 'FactValueTranslationID', 'FactComments', 'Language', 'DateModified']
Columns used: code='SpatialDimValueCode' -> ISO3 | year='Period' -> Year | value='FactValueNumeric' -> UHC_Service_Coverage_Index
Distinct values in 'Indicator': ['UHC Service Coverage Index (SDG 3.8.1)']
Distinct values in 'IndicatorCode': ['UHC_INDEX_REPORTED']
Raw distinct country codes: 195
[UHC] rows with missing ISO3 or Year (excluded): 0
[UHC] values out

,ISO3,Country,Year,UHC_Service_Coverage_Index
0,AFG,Afghanistan,2000,30
1,AFG,Afghanistan,2001,31
2,AFG,Afghanistan,2002,32
3,AFG,Afghanistan,2003,33
4,AFG,Afghanistan,2004,34


## 4) Part 2 — Health Financing (GHED)

In [5]:
log("")
log("== PART 2: GHED (file 09) ==")
hdr = pd.read_excel(GHED_FILE, sheet_name="Data", nrows=0)
log(f"GHED raw columns: {hdr.shape[1]} (only the required ones are read)")

lower_to_orig = {str(c).strip().lower(): c for c in hdr.columns}
need = ["code", "year", "che_ppp_pc", "che_gdp", "oops_che"]
missing_cols = [c for c in need if c not in lower_to_orig]
if missing_cols:
    raise KeyError(f"Missing required GHED columns: {missing_cols}")
use = [lower_to_orig[c] for c in need]
if "location" in lower_to_orig:
    use.append(lower_to_orig["location"])   # للمراجعة فقط، اسم الدولة النهائي من الـDictionary

ghed_raw = pd.read_excel(GHED_FILE, sheet_name="Data", usecols=use)
ghed_raw.columns = [str(c).strip().lower() for c in ghed_raw.columns]
log(f"Raw rows: {len(ghed_raw)} | Raw distinct codes: {ghed_raw['code'].nunique()}")

RENAMES = {
    "code": "ISO3",
    "year": "Year",
    "che_ppp_pc": "Health_Exp_Per_Capita_PPP",
    "che_gdp": "Health_Exp_Pct_GDP",
    "oops_che": "OOP_Pct_Current_Health_Exp",
}
log(f"Columns renamed: {RENAMES}")
IND = ["Health_Exp_Per_Capita_PPP", "Health_Exp_Pct_GDP", "OOP_Pct_Current_Health_Exp"]

ghed = ghed_raw.rename(columns=RENAMES).copy()
if "location" in ghed.columns:
    ghed = ghed.rename(columns={"location": "location_raw"})
ghed["ISO3"] = ghed["ISO3"].astype("string").str.strip().str.upper()
ghed["Year"] = to_numeric_logged(ghed["Year"], "Year", "GHED")
for c in IND:
    ghed[c] = to_numeric_logged(ghed[c], c, "GHED")

bad_key = ghed["ISO3"].isna() | ghed["Year"].isna()
log(f"[GHED] rows with missing ISO3 or Year (excluded): {int(bad_key.sum())}")
ghed = ghed[~bad_key].copy()
if ((ghed["Year"] % 1) != 0).any():
    raise ValueError("Non-integer Year values found in GHED; review.")
ghed["Year"] = ghed["Year"].astype("int64")

# قيم غير منطقية
for c in ("Health_Exp_Per_Capita_PPP", "Health_Exp_Pct_GDP"):
    neg = ghed[ghed[c] < 0]
    log(f"[GHED] negative values in {c}: {len(neg)}")
    if len(neg):
        log(neg.head(30).to_string())
        if ON_OUT_OF_RANGE == "stop":
            raise ValueError(f"Negative values in {c}. Review before deciding.")
        ghed.loc[neg.index, c] = np.nan

oop_bad = ghed[(ghed["OOP_Pct_Current_Health_Exp"] < 0) | (ghed["OOP_Pct_Current_Health_Exp"] > 100)]
log(f"[GHED] OOP values outside 0-100: {len(oop_bad)}")
if len(oop_bad):
    log(oop_bad.head(30).to_string())
    if ON_OUT_OF_RANGE == "stop":
        raise ValueError("OOP_Pct_Current_Health_Exp outside 0-100. Review before deciding.")
    ghed.loc[oop_bad.index, "OOP_Pct_Current_Health_Exp"] = np.nan

gdp_high = int((ghed["Health_Exp_Pct_GDP"] > 100).sum())
log(f"[GHED] Health_Exp_Pct_GDP > 100 (flag only, not modified): {gdp_high}")

ghed, ghed_unmapped = attach_country(ghed, "GHED", name_col="location_raw" if "location_raw" in ghed.columns else None)
ghed = resolve_duplicates(ghed, IND, "GHED")

all_null = int(ghed[IND].isna().all(axis=1).sum())
log(f"[GHED] rows where all 3 indicators are NaN (kept, not filled): {all_null}")

health_financing = (ghed[["ISO3", "Country", "Year"] + IND]
                    .sort_values(["ISO3", "Year"]).reset_index(drop=True))
log(f"[GHED] rows before: {len(ghed_raw)} | after: {len(health_financing)}")
health_financing.head()


== PART 2: GHED (file 09) ==
GHED raw columns: 4120 (only the required ones are read)
Raw rows: 4612 | Raw distinct codes: 195
Columns renamed: {'code': 'ISO3', 'year': 'Year', 'che_ppp_pc': 'Health_Exp_Per_Capita_PPP', 'che_gdp': 'Health_Exp_Pct_GDP', 'oops_che': 'OOP_Pct_Current_Health_Exp'}
[GHED] rows with missing ISO3 or Year (excluded): 0
[GHED] negative values in Health_Exp_Per_Capita_PPP: 0
[GHED] negative values in Health_Exp_Pct_GDP: 0
[GHED] OOP values outside 0-100: 0
[GHED] Health_Exp_Pct_GDP > 100 (flag only, not modified): 0
[GHED] ISO3 codes in data: 195 | mapped to dictionary: 195 | NOT in dictionary: 0
[GHED] Duplicate ISO3+Year rows: 0
[GHED] rows where all 3 indicators are NaN (kept, not filled): 2
[GHED] rows before: 4612 | after: 4612


,ISO3,Country,Year,Health_Exp_Per_Capita_PPP,Health_Exp_Pct_GDP,OOP_Pct_Current_Health_Exp
0,AFG,Afghanistan,2002,85.857495,9.443391,85.375603
1,AFG,Afghanistan,2003,85.933025,8.941258,86.069191
2,AFG,Afghanistan,2004,93.935804,9.808474,84.527588
3,AFG,Afghanistan,2005,105.927706,9.948289,78.970085
4,AFG,Afghanistan,2006,118.405820,10.622766,76.824974


## 5) Validation + Coverage Summary

In [6]:
log("")
summarize(healthcare_coverage, ["UHC_Service_Coverage_Index"], "healthcare_coverage_clean")
summarize(health_financing, IND, "health_financing_clean")

log("")
log("Unmapped codes -> UHC: " + (str(uhc_unmapped) if uhc_unmapped else "none"))
log("Unmapped codes -> GHED: " + (str(ghed_unmapped) if ghed_unmapped else "none"))

cov = pd.concat([
    coverage_long(healthcare_coverage, ["UHC_Service_Coverage_Index"]),
    coverage_long(health_financing, IND),
], ignore_index=True)

cov_wide = cov.pivot(index="Year", columns="Indicator", values="Number_of_Countries_With_Data")
log("")
log("Coverage Summary (Number_of_Countries_With_Data per Year; blank = no data):")
log(cov_wide.to_string())
cov_wide


--- Validation summary: healthcare_coverage_clean ---
Rows: 4680 | Countries: 195 | Duplicates on ISO3+Year: 0
UHC_Service_Coverage_Index: non-null=4680 | missing=0 | min year=2000 | max year=2023
--- Validation summary: health_financing_clean ---
Rows: 4612 | Countries: 195 | Duplicates on ISO3+Year: 0
Health_Exp_Per_Capita_PPP: non-null=4609 | missing=3 | min year=2000 | max year=2024
Health_Exp_Pct_GDP: non-null=4610 | missing=2 | min year=2000 | max year=2024
OOP_Pct_Current_Health_Exp: non-null=4610 | missing=2 | min year=2000 | max year=2024

Unmapped codes -> UHC: none
Unmapped codes -> GHED: none

Coverage Summary (Number_of_Countries_With_Data per Year; blank = no data):
Indicator  Health_Exp_Pct_GDP  Health_Exp_Per_Capita_PPP  OOP_Pct_Current_Health_Exp  UHC_Service_Coverage_Index
Year                                                                                                            
2000                    186.0                      186.0                       186.0

Indicator,Health_Exp_Pct_GDP,Health_Exp_Per_Capita_PPP,OOP_Pct_Current_Health_Exp,UHC_Service_Coverage_Index
Year,,,,
2000,186.0,186.0,186.0,195.0
2001,186.0,186.0,186.0,195.0
2002,187.0,187.0,187.0,195.0
2003,189.0,189.0,189.0,195.0
2004,189.0,189.0,189.0,195.0
2005,189.0,189.0,189.0,195.0
2006,189.0,189.0,189.0,195.0
2007,189.0,189.0,189.0,195.0
2008,189.0,189.0,189.0,195.0


## 6) Final checks + save

In [7]:
assert healthcare_coverage.duplicated(["ISO3", "Year"]).sum() == 0
assert health_financing.duplicated(["ISO3", "Year"]).sum() == 0
assert healthcare_coverage["Year"].dtype == "int64" and health_financing["Year"].dtype == "int64"
assert list(healthcare_coverage.columns) == ["ISO3", "Country", "Year", "UHC_Service_Coverage_Index"]
assert list(health_financing.columns) == ["ISO3", "Country", "Year"] + IND

healthcare_coverage.to_csv(OUT_DIR / "healthcare_coverage_clean.csv", index=False, encoding="utf-8-sig")
health_financing.to_csv(OUT_DIR / "health_financing_clean.csv", index=False, encoding="utf-8-sig")
cov.to_csv(OUT_DIR / "coverage_summary.csv", index=False, encoding="utf-8-sig")

# إعادة قراءة المخرجات للتأكد
chk1 = pd.read_csv(OUT_DIR / "healthcare_coverage_clean.csv")
chk2 = pd.read_csv(OUT_DIR / "health_financing_clean.csv")
assert chk1.shape == healthcare_coverage.shape and chk2.shape == health_financing.shape
assert chk1.duplicated(["ISO3", "Year"]).sum() == 0 and chk2.duplicated(["ISO3", "Year"]).sum() == 0
log("")
log("Final: Duplicate ISO3 + Year = 0 in both outputs. Files saved.")

(OUT_DIR / "cleaning_log.txt").write_text("\n".join(LOG), encoding="utf-8")
print("cleaning_log.txt written")


Final: Duplicate ISO3 + Year = 0 in both outputs. Files saved.
cleaning_log.txt written
